In [14]:
# Load packages
import pandas as pd
import numpy  as np
import webbrowser
import re

In [2]:
# Define categories that I care for
key_categories = [
                #   "astro-ph.CO", # Cosmology and Nongalactic Astrophysics
                #   "astro-ph.EP", # Earth and Planetary Astrophysics
                  "astro-ph.GA", # Astrophysics of Galaxies
                  "astro-ph.HE", # High Energy Astrophysical Phenomena
                #   "astro-ph.IM", # Instrumentation and Methods for Astrophysics
                #   "astro-ph.SR", # Solar and Stellar Astrophysics
                  ]

# Define keywords to search for
key_words = ["diffusion",
             "diffuse",
             "propagation",
             "gamma",
             "ɣ",
             "γ",
             "cosmic",
             "neutrino",
             "Milky Way",
             "Galactic",
             "H.E.S.S.",
             "LHAASO",
             "Tibet",
             "ARGO",
             "IceCube",
             ]

# Define authors to search for
key_authors = ["Porter",
               "Rowell",
               "Moskalenko",
               "Einecke",
               "Mertsch",
               "Vechiotti",
               "Alsulami",
               "Collins",
               "Feijen",
               "Capecchiacci",
               "Lopez",
               "Lange",
               ]

In [ ]:
def load_list(filename):
    """
    Load search terms from a text file. These are used for regex searches, so word boundaries are added
    """
    items = []

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            items.append(r"\b"+line+r"\b")

    return items

key_authors     = load_list("./key_authors.txt")
key_words       = load_list("./key_words.txt")
exclusion_words = load_list("./exclusion_words.txt")

In [10]:
key_authors

['\\bPorter\\b',
 '\\bRowell\\b',
 '\\bMoskalenko\\b',
 '\\bEinecke\\b',
 '\\bMertsch\\b',
 '\\bSchwefer\\b',
 '\\bVecchiotti\\b',
 '\\bMitchell\\b',
 '\\bAlsulami\\b',
 '\\bKoenig\\b',
 '\\bCollins\\b',
 '\\bFeijen\\b',
 '\\bCapecchiacci\\b',
 '\\bLopez\\b',
 '\\bLange\\b']

In [11]:
key_words

['\\bdiffusion\\b',
 '\\bdiffuse\\b',
 '\\bpropagation\\b',
 '\\bCR\\b',
 '\\bCRs\\b',
 '\\bgamma\\b',
 '\\bɣ\\b',
 '\\bγ\\b',
 '\\bcosmic\\b',
 '\\bneutrino\\b',
 '\\bMilky Way\\b',
 '\\bGalactic\\b',
 '\\bH.E.S.S.\\b',
 '\\bHESS\\b',
 '\\bLHAASO\\b',
 '\\bTibet\\b',
 '\\bARGO\\b',
 '\\bIceCube\\b']

In [12]:
exclusion_words

['\\bextragalactic\\b',
 '\\bextra-galactic\\b',
 '\\bhigh-z\\b',
 '\\bquasar\\b',
 '\\bblazar\\b',
 '\\bGRB\\b',
 '\\bGRBs\\b',
 '\\bAGN\\b',
 '\\bAGNs\\b',
 '\\bhigh-redshift\\b']

In [23]:
# fill = len(max(key_authors, key=len))
fill = len( max(key_authors, key=len) ) - 4
fill

12

In [19]:
a = re.search("\\bAGN\\b", "agnostic", re.IGNORECASE)
print(a)

None


In [143]:
def LaTeX_to_unicode(s):
    """Stip LaTeX-style ligatures/special letters and accents from the string (e.g. {\'a} -> a, and \'a -> a)
    """

    if s is None:
        return ""
    
    # Dictionary of the LaTeX accents
    # Same ordering as on the wikipedia page
    LATEX_ACCENTS = {
                    "`": "\u0300",   # grave,                              e.g. ò
                    "'": "\u0301",   # acute,                              e.g. ó
                    "^": "\u0302",   # circumflex,                         e.g. ô
                    '"': "\u0308",   # umlaut, trema, or dieresis,         e.g. ö
                    # "H": "\u030B",   # long Hungarian umlaut/double acute, e.g. ő
                    "~": "\u0303",   # tilde,                              e.g. õ
                    "c": "\u0327",   # cedilla,                            e.g. ç
                    "k": "\u0328",   # ogonek,                             e.g. ą
                                     # barred l,                           e.g. ł
                    "=": "\u0304",   # macron,                             e.g. ō
                                     # under-bar,                          e.g. o
                    # ".": "\u0307",   # dot,                                e.g. ȯ
                                     # under-dot,                          e.g. ụ
                    "r": "\u030A",   # ring,                               e.g. å
                                     # ringed a (special case),            e.g. å
                    "u": "\u0306",   # breve,                              e.g. ŏ
                    "v": "\u030C",   # caron,                              e.g. š
                                     # tie,                                e.g. o͡o
                                     # slashed o,                          e.g. ø
                                     # dotless i,                          e.g. ı
                    }
    
    # LATEX_LIGATURES = {
    #                   r"\ae": "ae",
    #                   r"\AE": "AE",
    #                   r"\oe": "oe",
    #                   r"\OE": "OE",
    #                   r"\aa": "aa",
    #                   r"\AA": "AA",
    #                   r"\o":  "o",
    #                   r"\O":  "O",
    #                   r"\ss": "ss",
    #                   r"\l":  "l",
    #                   r"\L":  "L",
    #                   r"\i":  "i",
    #                   r"\j":  "j",
    #                   }
    
    # # Ligatures and special letters
    # for latex, repl in LATEX_LIGATURES.items():
    #     s = re.sub(latex + r"\b", repl, s)

    # {\'a} style
    for latex, combining in LATEX_ACCENTS.items():
        s = re.sub(
                  rf"\{{{latex}([A-Za-z])\}}",
                  lambda m: m.group(1) + combining,
                  s,
                  )

    # \'a style
    for latex, combining in LATEX_ACCENTS.items():
        s = re.sub(
                  rf"{latex}([A-Za-z])",
                  lambda m: m.group(1) + combining,
                  s,
                  )

    # # Handle {\'o} style
    # s = re.sub(
    #     r"\{\\([`'^\"~c])\s*([A-Za-z])\}",
    #     lambda m: m.group(2) + LATEX_ACCENTS[m.group(1)],
    #     s,
    # )

    # # Handle \'o style
    # s = re.sub(
    #     r"\\([`'^\"~c])\s*([A-Za-z])",
    #     lambda m: m.group(2) + LATEX_ACCENTS[m.group(1)],
    #     s,
    # )

    return s


In [144]:
import unicodedata

def normalise_string(s):
    """Normalise a string, i.e. remove accents (e.g. ó -> o)
    Also accounts for LaTeX accents
    """

    # Define the form
    form = "NFKD" # "compatibility deecomposition"

    if s is None:
        return ""
    
    s_stripped   = LaTeX_to_unicode(s)
    s_normalised = unicodedata.normalize(form, s_stripped)
    
    return s_normalised.encode("ascii", "ignore").decode("ascii")


In [145]:
test = "L\'opez, L{\'o}pez, S\\o rensen, H\\aa kon, \\L ukasz, Garc\\'ia, Bla\\c{s}kovi\\'c, \\i vanov, o\\underbar{o}, u\\d{u}, o\\tie{o}, ð, þ, ł, ø, ı"

In [146]:
normalise_string("Lángé, A. and L{\'o}pez, S. and J\'ohannesson, G. and Hello World")

'Lange, A. and Lopez, S. and Johannesson, G. and Hello Wold'

In [105]:
normalise_string(test)

'L\\opez, L{\\o}pez, S\\o rensen, H\\aa kon, \\L ukasz, Garc\\ia, Bla\\c{s}kovi\\c, \\i vanov, o\\underbar{o}, u\\d{u}, o\\tie{o}, , , , , '

In [255]:
# Define filename
# filename = "./catchuptest.txt"
filename = "./catchup.txt"

# Load the file
with open(filename, "r", encoding="utf-8") as f:
    text = f.read()

# Split by entry boundaries: assume entries between --- ... ---
raw_entries = re.split(r"\n------------------------------------------------------------------------------\n", text)
data = []

# print(raw_entries)

# Extract blocks of text
count_papers = 0
for block in raw_entries:
    count_papers += 1
    block = block.strip()
    # print(block, "\n\n")
    if not block: # not sure what this does but it is required
        continue

    # Only keep the block if it contains an arXiv number
    if "arXiv:" not in block:
        continue

    # print(block, "\n\n")

    # Initialise dictionary
    entry = {}

    # Extract sections
    entry["arXiv Number"] = re.search(rf"^arXiv:\s*(.*)",    block, re.MULTILINE).group(1).strip()
    entry["Title"]        = re.search(rf"^Title: \s*(.*)",   block, flags=re.MULTILINE).group(1).strip()
    entry["Authors"]      = re.search(rf"^Authors: \s*(.*)", block, flags=re.MULTILINE).group(1).strip()
    entry["Categories"]   = re.search(rf"^Categories: \s*(.*)", block, flags=re.MULTILINE).group(1).strip()
    
    # Extract if it is a revised version
    revised_match = re.search(rf"^replaced with revised version", block, flags=re.MULTILINE)
    entry["Revised?"] = True if revised_match else False
    
    # print(entry)

    # Extract the abstract, split across multple lines between `\\\n  ' and `\\ ( https://arxiv.org'
    # Account for the cases where there is no abstract (e.g. revised versions)
    abstract_match = re.search(r"\\\\\n  \s*(.*?)\s*\\\\\s*\(", block, re.DOTALL)
    entry["Abstract"] = abstract_match.group(1).strip() if abstract_match else None
    
    # print(entry)

    # Extract url inside \\ ( url , size )
    uv_match = re.search(r"\\\\\s*\( \s*([^,]+)\s*,\s*([^)]+)\)", block)
    entry["url"] = uv_match.group(1).strip()
    # entry["url_value"] = uv_match.group(2).strip()
    
    # print(entry)

    # Add a currently empty column for match reasons
    entry["Matches"] = None

    # # Add entry to the data
    # data.append(entry)

    # skip entry if it is a revised version

    # Add entry to the data *ONLY IF* it is from one of the categpries I care about
    # Doing this at the end is slower but who cares it is easier to read this way
    # print(entry)
    for key_category in key_categories:

        category_match = re.search(key_category, entry["Categories"])
        # print(category_match)

        if category_match:

            # print(entry["arXiv Number"])
            data.append(entry)
            # print("")

            # If a match is found in one of the key categories, break back into the main for loop
            break

# print(data)

# Place into a dataframe
df = pd.DataFrame(data)
# print(df)

In [256]:
print(len(raw_entries))
print(raw_entries)

2782
['------------------------------------------------------------------------------', 'Send any comments regarding submissions directly to submitter.', 'Archives at http://arxiv.org/\nTo unsubscribe, e-mail To: astro-ph@arXiv.org, Subject: cancel', 'received from  Tue 15 Jul 25 18:00:00 GMT  to  Wed 16 Jul 25 18:00:00 GMT', "\\\\\narXiv:2507.11581\nDate: Tue, 15 Jul 2025 09:44:29 GMT   (6298kb)\n\nTitle: SAMPLE -- Stratospheric Altitude Microbiology Probe for Life Existence\n  -- A Method of Collection of Stratospheric Samples Using Balloon-Borne\n  Payload System\nAuthors: Margarita Safonova, Bharat Chandra P, Binukumar G. Nair, Akshay Datey,\n  Dipshikha Chakravortty, Ajin Prakash, Mahesh Babu, Shubham Ghatul, Shubhangi\n  Jain, Rekhesh Mohan, Jayant Murthy\nCategories: astro-ph.IM astro-ph.EP\n\\\\\n  The Earth possesses many environmental extremes that mimic conditions on\nextraterrestrial worlds. The stratosphere at 30-40 km altitude closely\nresembles the surface of Mars in ter

In [ ]:
# Delete ` (*cross-listing*)' from the arXiv numbers
for entry_count in range(0, len(df)):
    # print(df.loc[entry_count, "arXiv Number"])
    if " (*cross-listing*)" in df.loc[entry_count, "arXiv Number"]:
        # print("          ^^^here")
        df.loc[entry_count, "arXiv Number"] = df.loc[entry_count, "arXiv Number"].replace(" (*cross-listing*)", "")
    # print(df.loc[entry_count, "arXiv Number"])
    # print("")


# Remove duplicated arXiv numbers
print(len(df))
df.drop_duplicates(subset="arXiv Number", inplace=True, ignore_index=True)
print(len(df))

# Remove revised papers
# print(df)
print(len(df))
df.drop(df[df["Revised?"]==True].index, inplace=True)
df.reset_index(drop=True, inplace=True)
# print(df)
print(len(df))

# df.head(48)

1538
1430
1430
1003


,arXiv Number,Title,Authors,Categories,Revised?,Abstract,url,Matches
0,2507.11600,The ALMA-CRISTAL survey: Resolved kinematic st...,"Lilian L. Lee, Natascha M. F\""orster Schreiber...",astro-ph.GA,False,We present a detailed kinematic study of a sam...,https://arxiv.org/abs/2507.11600,None
1,2507.11602,Unmixed metals: Variations in the enrichment o...,"Trystyn A. M. Berg, Louise A. Welsh, Ryan J. C...",astro-ph.GA,False,The chemical abundance patterns of near-pristi...,https://arxiv.org/abs/2507.11602,None
2,2507.11603,THOR: a GPU-accelerated and MPI-parallel radia...,"Chris Byrohl, Dylan Nelson",astro-ph.GA astro-ph.IM,False,Emission and absorption line features are impo...,https://arxiv.org/abs/2507.11603,None
3,2507.11605,High-redshift AGN population in radiation-hydr...,"Teodora-Elena Bulichi, Oliver Zier, Aaron Smit...",astro-ph.GA,False,High-redshift active galactic nuclei (AGN) hav...,https://arxiv.org/abs/2507.11605,None
4,2507.11609,Stephenson 2 DFK 52: Discovery of an exotic re...,"Mark A. Siebert, Elvire De Beck, Guillermo Qui...",astro-ph.GA astro-ph.SR,False,Atacama Large Millimeter/submillimeter Array (...,https://arxiv.org/abs/2507.11609,None
5,2507.11613,COS-EDGES: Co-rotation and Kinematic Stratific...,"Glenn G. Kacprzak, Benjamin D. Oppenheimer, Ni...",astro-ph.GA,False,We present the first results from the COS-EDGE...,https://arxiv.org/abs/2507.11613,None
6,2507.11616,A Hybrid Algorithm for Drift-Kinetic Particle ...,"Tyler Trent, Dimitrios Psaltis, Feryal \""Ozel",astro-ph.HE physics.comp-ph physics.plasm-ph,False,Astrophysical plasmas in relativistic spacetim...,https://arxiv.org/abs/2507.11616,None
7,2507.11629,DELVE-ing into the Milky Way's Globular Cluste...,"A. Chiti, K. Tavangar, P. S. Ferguson, J. A. C...",astro-ph.GA,False,Extra-tidal features around globular clusters ...,https://arxiv.org/abs/2507.11629,None
8,2507.11635,Efficacy of Galaxy Catalogues for following up...,"Tamojeet Roychowdhury, Harsh Choudhary, Varun ...",astro-ph.HE,False,The detection of gravitational waves (GW) by t...,https://arxiv.org/abs/2507.11635,None
9,2507.11647,X-ray Measurements of $^{44}$Ti in Four Supern...,Tyler E. Hanover and Mark D. Leising,astro-ph.HE,False,We present the results of a search for scandiu...,https://arxiv.org/abs/2507.11647,None


In [252]:
df.loc[46]

arXiv Number                                           2507.12227
Title           Dark Matter Clumps as Sources of Gravitational...
Authors         Ezequiel Alvarez, Scott Perkins, Federico Rava...
Categories                               gr-qc astro-ph.HE hep-ph
Revised?                                                    False
Abstract        We consider the hypothetical possibility that ...
url                              https://arxiv.org/abs/2507.12227
Matches                                                      None
Name: 46, dtype: object

In [176]:
def add_match(df, row_index, match_text):

    # Add match to the df
    # Append the match reason to the df. If there is not currently a reason, then overwrite the None
    if df["Matches"][row_index] is None:
        df.loc[row_index, "Matches"] = match_text
    else:
        # df["Matches"][row_index].append(", ", match_text)
        df.loc[row_index, "Matches"] = df.loc[row_index, "Matches"] + ", " + match_text

    return

In [ ]:
entries_of_note = []

# Loop over all entries
for entry_count in range(0, len(df)):

    # print(df["arXiv Number"].loc[entry_count])
    
    # Search all author lists for the people I care about
    for key_author in key_authors:

        # Search the author field in the entry
        author_match = re.search(key_author, df["Authors"][entry_count])
        # print(re.search(key_author, df["Authors"][entry_count]))
        if author_match:
            # print(entry_count, author_match)
            # print(df.loc[entry_count])
            entries_of_note.append(entry_count)
            # print(author_match)

            # print(df["Matches"][entry_count])
            # df["Matches"][entry_count] = key_author
            # print(df["Matches"][entry_count])
            add_match(df, entry_count, key_author)

            # print("")
    
    # Search all titles and abstracts for words that I care about
    for key_word in key_words:

        # Search the author field in the entry
        title_match    = re.search(key_word, df["Title"][entry_count], re.IGNORECASE)
        if df["Abstract"][entry_count] is not None: # Skip empty abstract entries
            abstract_match = re.search(key_word, df["Abstract"][entry_count], re.IGNORECASE)
        # print(re.search(key_author, df["Authors"][entry_count]))
        # print(title_match)
        # print(abstract_match)
        if title_match or abstract_match:
            # print(entry_count, author_match)
            # print(df.loc[entry_count])
            entries_of_note.append(entry_count)
            # print(title_match)
            # print(abstract_match)
            
            # print(df["Matches"][entry_count])
            # df["Matches"][entry_count] = key_word
            add_match(df, entry_count, key_word)

            # print("")

# print(entries_of_note)

# Only keep unique entries
entries_of_note_unique = np.unique(entries_of_note)
# print(entries_of_note_unique)

# print(len(entries_of_note_unique))

[ 2  3  5  7  9 10 11 15 18 23 26 27 28 30 32 34 37 38 39 40 41 49 65 69
 72]
25


In [178]:
print(df["url"].loc[entries_of_note_unique], df["Matches"].loc[entries_of_note_unique])

2     https://arxiv.org/abs/2507.11603
3     https://arxiv.org/abs/2507.11605
5     https://arxiv.org/abs/2507.11613
7     https://arxiv.org/abs/2507.11629
9     https://arxiv.org/abs/2507.11647
10    https://arxiv.org/abs/2507.11658
11    https://arxiv.org/abs/2507.11664
15    https://arxiv.org/abs/2507.11741
18    https://arxiv.org/abs/2507.11774
23    https://arxiv.org/abs/2507.11829
26    https://arxiv.org/abs/2507.11993
27    https://arxiv.org/abs/2507.12046
28    https://arxiv.org/abs/2507.12074
30    https://arxiv.org/abs/2507.12120
32    https://arxiv.org/abs/2507.12209
34    https://arxiv.org/abs/2507.12222
37    https://arxiv.org/abs/2507.12275
38    https://arxiv.org/abs/2507.12316
39    https://arxiv.org/abs/2507.12374
40    https://arxiv.org/abs/2507.12405
41    https://arxiv.org/abs/2506.19910
49    https://arxiv.org/abs/2410.22976
65    https://arxiv.org/abs/2506.01679
69    https://arxiv.org/abs/2507.01942
72    https://arxiv.org/abs/2507.08526
Name: url, dtype: object 

In [181]:
webbrowser.open("https://arxiv.org/abs/2507.08526")

True

In [ ]:
for link_index in entries_of_note_unique:
    link = df.loc[link_index, "url"]
    # print(link)
    webbrowser.open(link)

https://arxiv.org/abs/2507.11603
https://arxiv.org/abs/2507.11605
https://arxiv.org/abs/2507.11613
https://arxiv.org/abs/2507.11629
https://arxiv.org/abs/2507.11647
https://arxiv.org/abs/2507.11658
https://arxiv.org/abs/2507.11664
https://arxiv.org/abs/2507.11741
https://arxiv.org/abs/2507.11774
https://arxiv.org/abs/2507.11829
https://arxiv.org/abs/2507.11993
https://arxiv.org/abs/2507.12046
https://arxiv.org/abs/2507.12074
https://arxiv.org/abs/2507.12120
https://arxiv.org/abs/2507.12209
https://arxiv.org/abs/2507.12222
https://arxiv.org/abs/2507.12275
https://arxiv.org/abs/2507.12316
https://arxiv.org/abs/2507.12374
https://arxiv.org/abs/2507.12405
https://arxiv.org/abs/2506.19910
https://arxiv.org/abs/2410.22976
https://arxiv.org/abs/2506.01679
https://arxiv.org/abs/2507.01942
https://arxiv.org/abs/2507.08526


In [2]:
# Define categories that I care for
key_categories = [
                #   "astro-ph.CO", # Cosmology and Nongalactic Astrophysics
                #   "astro-ph.EP", # Earth and Planetary Astrophysics
                  "astro-ph.GA", # Astrophysics of Galaxies
                  "astro-ph.HE", # High Energy Astrophysical Phenomena
                #   "astro-ph.IM", # Instrumentation and Methods for Astrophysics
                #   "astro-ph.SR", # Solar and Stellar Astrophysics
                  ]

In [5]:
cat_string = "+OR+".join(f"cat:{c}" for c in key_categories)
cat_string

'cat:astro-ph.GA+OR+cat:astro-ph.HE'

In [7]:
import urllib, urllib.request
import time
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET

In [30]:
filename = "/Users/pmarinos/Documents/PYTHON/arXiv/catchup.txt"
with open(filename, "r", encoding="utf-8") as f:
    text = [next(f).rstrip("\n") for _ in range(3)]

start_date = datetime(year=int(text[0]), month=int(text[1]), day=int(text[2])).astimezone(timezone.utc) - timedelta(days=1)
start_date

datetime.datetime(2026, 2, 1, 8, 0, tzinfo=datetime.timezone.utc)

In [31]:
# Obtain the current date
current_time = datetime.now(timezone.utc)
# The list of papers is typically released before 06:00 UTC.
# If executing before 06:00 UTC, set the date to one day prior
if current_time.hour < 6:
    current_time = current_time - timedelta(days=1)
# The lists are published for the previous day. Always go back one day in the search.
dt = 1
# No lists are published over the weekend. If it is Sunday, go back one extra day in the search, and two days for Monday.
current_weekday = current_time.weekday()
if current_weekday == 6:
    dt += 1
elif current_weekday == 0:
    dt += 2
# No lists are released on certain days. These days are chosen ad-hoc, and are days that are important to USAians. It includes Christmas, their Thanksgiving, and others.
# The search should return no results on those days (not tested).
# If waiting extra time, there should be no missed papers (not tested).
# Compute the end_date of the search
end_date = current_time - timedelta(days=dt)

In [32]:
end_date >= start_date

True

In [33]:
(end_date - start_date)

datetime.timedelta(days=1, seconds=48755, microseconds=182448)

In [34]:
(end_date - start_date).days

1

In [38]:
# Namespaces used by arXiv
ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "opensearch": "http://a9.com/-/spec/opensearch/1.1/",
    "arxiv": "http://arxiv.org/schemas/atom",
}

In [ ]:
# arXiv asks for a 3-second pause between searches of ten papers
sleep_time = 3
interval   = 10

url = "https://export.arxiv.org/api/query?search_query=submittedDate:[{start_year:d}{start_month:02d}{start_day:02d}1900%20TO%{end_year:d}{end_month:02d}{end_day:02d}1900]+AND+{cats:s}&sortBy=submittedDate&start={start_num:d}&max_results={end_num:d}"

# Perform the search term to extract how many papers there are
formatted_url_initial = url.format(start_year  = start_date.year,
                           start_month = start_date.month,
                           start_day   = start_date.day,
                           end_year    = end_date.year,
                           end_month   = end_date.month,
                           end_day     = end_date.day,
                           cats        = cat_string,
                           start_num   = 0,
                           end_num     = 1)
with urllib.request.urlopen(formatted_url_initial) as f:
    xml_data_initial = f.read()
    
parsed_xml_data_initial = ET.fromstring(xml_data_initial)

max_num = int(parsed_xml_data_initial.find("opensearch:totalResults", ns).text)
# replace max_num with a small number for testing
max_num  = 3

print("Searching for papers. Estimated time: {:d} seconds".format(sleep_time*max_num//interval + (sleep_time if max_num%interval > 0 else 0)))

# Loop over the searches
entries = []
request_count = 0
for ii in range(0, max_num, interval):

    if request_count > 0:
        time.sleep(sleep_time)
    request_count += 1

    formatted_url = url.format(start_year  = start_date.year,
                               start_month = start_date.month,
                               start_day   = start_date.day,
                               end_year    = end_date.year,
                               end_month   = end_date.month,
                               end_day     = end_date.day,
                               cats        = cat_string,
                               start_num   = ii,
                               end_num     = ii+interval)

    with urllib.request.urlopen(formatted_url) as f:
        xml_data = f.read()

    parsed_xml = ET.fromstring(xml_data)
    
    # Loop over the entries (papers) within the current search
    for entry in parsed_xml.findall("atom:entry", ns):

        published_date = entry.find("atom:published", ns).text
        updated_date   = entry.find("atom:updated", ns).text

        author_list = [author.find("atom:name", ns).text for author in entry.findall("atom:author", ns)]

        paper = {
            "arXiv Number": entry.find("atom:id", ns).text.split("/")[-1],
            "Title": entry.find("atom:title", ns).text.strip(),
            "Authors": ", ".join(f"{author}" for author in author_list),
            # "categories": [
            #     cat.attrib["term"]
            #     for cat in entry.findall("atom:category", ns)
            # ],
            "Revised?": updated_date > published_date,
            "Abstract": entry.find("atom:summary", ns).text.strip(),
            "url": entry.find("atom:id", ns).text.strip(),
            # "published": entry.find("atom:published", ns).text,
            # "updated": entry.find("atom:updated", ns).text,
            # "primary_category": entry.find(
            #     "arxiv:primary_category", ns
            # ).attrib["term"],
            "Matches": None
        }

        entries.append(paper)

# print(entries)

Searching for papers. Estimated time: 3 seconds


In [79]:
# xml_data

In [80]:
# entries

In [90]:
df = pd.DataFrame(entries)

In [83]:
with open("catchup.txt", "w") as f:
    f.write(f"{end_date.year:d}\n")
    f.write(f"{end_date.month:d}\n")
    f.write(f"{end_date.day:d}\n")

In [91]:
df

,arXiv Number,Title,Authors,Revised?,Abstract,url,Matches
0,2601.23282v1,PDRs4All: XVIII. The evolution of the PAH ioni...,"Alexandros Maragkoudakis, Christiaan Boersma, ...",False,We investigate the evolution of the PAH popula...,http://arxiv.org/abs/2601.23282v1,None
1,2601.23264v1,MARVELously Dark: the gravothermal evolution o...,"Anna Engelhardt, Ferah Munshi, Annika H. G. Pe...",False,Self-interacting dark matter (SIDM) with a suf...,http://arxiv.org/abs/2601.23264v1,None
2,2601.23260v1,Evolution of Supermassive Black Hole Pairs on ...,"Sena Ghobadi, David Ballantyne, Tamara Bogdanovic",False,Theoretical models of the evolution of superma...,http://arxiv.org/abs/2601.23260v1,None
3,2601.23250v1,Too many or too massive? Investigating the hig...,"Daniel Roberts, Francesco Shankar, Vieri Camme...",False,Recent JWST observations have unveiled a numer...,http://arxiv.org/abs/2601.23250v1,None
4,2601.23242v1,Physical origin of very-high-energy gamma rays...,"Shilong Chen, Abhishek Das, B. Theodore Zhang,...",False,Relativistic jets in active galactic nuclei (A...,http://arxiv.org/abs/2601.23242v1,None
5,2601.23205v1,Human versus Artificial Inteligence; a signifi...,A. De Rújula,False,There are two well documented models of gamma ...,http://arxiv.org/abs/2601.23205v1,None
6,2601.23069v1,Quantifying the C/O ratio in the planet-formin...,"Javiera K. Díaz-Berríos, Catherine Walsh, Ewin...",False,The material in planet-forming disks determine...,http://arxiv.org/abs/2601.23069v1,None
7,2601.23013v1,Contrastive Learning of Extragalactic Stellar ...,"Ernesto Benitez-Walz, Jelle Mes, Juan Miró-Car...",False,We present a self-supervised approach for char...,http://arxiv.org/abs/2601.23013v1,None
8,2601.22972v1,G183: An outer galaxy filament feeding a massi...,"Bhaswati Mookerjea, Saurav Sen, V. S. Veena, C...",False,We present the first detailed multi-tracer obs...,http://arxiv.org/abs/2601.22972v1,None
9,2601.22914v1,Exact black holes and black branes with bumpy ...,"Fabrizio Canfora, Andrés Gomberoff, Carla Henr...",False,We present exact solutions of the Einstein-$SU...,http://arxiv.org/abs/2601.22914v1,None


In [92]:
df.drop(index=3, inplace=True)

In [93]:
df

,arXiv Number,Title,Authors,Revised?,Abstract,url,Matches
0,2601.23282v1,PDRs4All: XVIII. The evolution of the PAH ioni...,"Alexandros Maragkoudakis, Christiaan Boersma, ...",False,We investigate the evolution of the PAH popula...,http://arxiv.org/abs/2601.23282v1,None
1,2601.23264v1,MARVELously Dark: the gravothermal evolution o...,"Anna Engelhardt, Ferah Munshi, Annika H. G. Pe...",False,Self-interacting dark matter (SIDM) with a suf...,http://arxiv.org/abs/2601.23264v1,None
2,2601.23260v1,Evolution of Supermassive Black Hole Pairs on ...,"Sena Ghobadi, David Ballantyne, Tamara Bogdanovic",False,Theoretical models of the evolution of superma...,http://arxiv.org/abs/2601.23260v1,None
4,2601.23242v1,Physical origin of very-high-energy gamma rays...,"Shilong Chen, Abhishek Das, B. Theodore Zhang,...",False,Relativistic jets in active galactic nuclei (A...,http://arxiv.org/abs/2601.23242v1,None
5,2601.23205v1,Human versus Artificial Inteligence; a signifi...,A. De Rújula,False,There are two well documented models of gamma ...,http://arxiv.org/abs/2601.23205v1,None
6,2601.23069v1,Quantifying the C/O ratio in the planet-formin...,"Javiera K. Díaz-Berríos, Catherine Walsh, Ewin...",False,The material in planet-forming disks determine...,http://arxiv.org/abs/2601.23069v1,None
7,2601.23013v1,Contrastive Learning of Extragalactic Stellar ...,"Ernesto Benitez-Walz, Jelle Mes, Juan Miró-Car...",False,We present a self-supervised approach for char...,http://arxiv.org/abs/2601.23013v1,None
8,2601.22972v1,G183: An outer galaxy filament feeding a massi...,"Bhaswati Mookerjea, Saurav Sen, V. S. Veena, C...",False,We present the first detailed multi-tracer obs...,http://arxiv.org/abs/2601.22972v1,None
9,2601.22914v1,Exact black holes and black branes with bumpy ...,"Fabrizio Canfora, Andrés Gomberoff, Carla Henr...",False,We present exact solutions of the Einstein-$SU...,http://arxiv.org/abs/2601.22914v1,None


In [98]:
df.loc[2] == True

arXiv Number    False
Title           False
Authors         False
Revised?        False
Abstract        False
url             False
Matches         False
Name: 2, dtype: bool

In [4]:
from datetime import timezone

In [5]:
datetime.now(timezone.utc).date()

NameError: name 'datetime' is not defined

In [6]:
datetime.now().date()

NameError: name 'datetime' is not defined

In [112]:
datetime.now().weekday()

0

In [ ]:
# PST
# 19:00 to 21:30, make it 22:00 for safety

# PST is at -8hr

# 22:00 + 08:00 = 06:00 UTC

In [3]:
if datetime.now(timezone.utc).hour < 6:
    end_date = datetime.now(timezone.utc) - timedelta(days=4)
else:
    end_date = datetime.now(timezone.utc) - timedelta(days=3)

end_date

NameError: name 'datetime' is not defined

In [106]:
datetime.now(timezone.utc) - timedelta(days=4)

datetime.datetime(2026, 1, 30, 1, 42, 35, 393241, tzinfo=datetime.timezone.utc)

In [ ]:
if before 06:00 GMT, subtract four days
if after 06:00 GMT, subtract three days

In [ ]:
==0
4
==6
3
==5
2
==1, 2, 3, 4
1

In [132]:
# current_time = datetime.now(timezone.utc)
current_time = datetime.now(timezone.utc) + timedelta(days=0, hours=5)

day_count = 1
if current_time.hour < 6:
    current_time = current_time - timedelta(days=1)
current_weekday = current_time.weekday()
print(current_weekday)
if current_weekday == 0:
    day_count += 2
elif current_weekday == 6:
    day_count += 1
elif current_weekday == 5:
    day_count += 0
print(day_count)
end_date = current_time - timedelta(days=day_count)
end_date

1
1


datetime.datetime(2026, 2, 2, 7, 37, 2, 529350, tzinfo=datetime.timezone.utc)

In [115]:
current_time

datetime.datetime(2026, 2, 3, 2, 7, 12, 968888, tzinfo=datetime.timezone.utc)

In [35]:
key_categories = [
                 # "astro-ph*", # All astrophysics categories
                 # "astro-ph.CO", # Cosmology and Nongalactic Astrophysics
                 # "astro-ph.EP", # Earth and Planetary Astrophysics
                #  "astro-ph.GA", # Astrophysics of Galaxies
                 "astro-ph.HE", # High Energy Astrophysical Phenomena
                 # "astro-ph.IM", # Instrumentation and Methods for Astrophysics
                 # "astro-ph.SR", # Solar and Stellar Astrophysics
                 ]
cat_string = "+OR+".join(f"cat:{c}" for c in key_categories)

cat_string

'cat:astro-ph.HE'